### PySpark Otomoto Demo 

Źródło danych: https://www.kaggle.com/datasets/szymoncyperski/car-sales-offers-from-otomotopl-2023 

In [ ]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

**Teoria:** Powyżej importujemy niezbędne biblioteki. `SparkSession` to główny punkt wejścia do funkcjonalności DataFrame i SQL w Sparku (od wersji 2.0). Moduł `functions` dostarcza wbudowane funkcje operujące na kolumnach, a `matplotlib.pyplot` posłuży nam do późniejszej wizualizacji danych.

In [ ]:
spark = SparkSession.builder \
    .appName("Otomoto Demo") \
    .getOrCreate()

**Teoria:** Tworzymy sesję Sparka. `builder` używa wzorca projektowego Builder do skonfigurowania sesji. `getOrCreate()` tworzy nową sesję lub pobiera istniejącą, co jest bezpieczne przy wielokrotnym uruchamianiu notatnika.

In [ ]:
df = spark.read.option("header", True) \
    .option("delimiter", ";") \
    .option("inferSchema", False) \
    .csv("/Workspace/Users/pichniarczykmarek@gmail.com/zad/otomoto_offers_eng_23-04-2023.csv")

**Teoria:** Wczytywanie danych. Spark używa leniwego ewaluowania (lazy evaluation) - dane nie są fizycznie wczytywane w tym momencie, tworzony jest tylko plan wykonania (DAG). Ustawiamy `header=True` ponieważ nasz plik CSV ma nagłówki, oraz określamy separator jako średnik `;`.

In [ ]:
df.show()

**Teoria:** `show()` to akcja (action), która uruchamia wykonanie obliczeń w Sparku. Dopiero teraz plik jest odczytywany, a wynik prezentowany na ekranie.

In [ ]:
df.filter(F.col("vehicle_brand") == "Volvo").show()

In [ ]:
df = df.withColumn("price_num",
                   F.regexp_replace(F.col("price"), r"[^\d]", "").cast("double"))

df = df.withColumn("mileage_km",
                   F.regexp_replace(F.col("mileage"), r"[^\d]", "").cast("integer"))

df = df.withColumn("production_year_int",
                   F.regexp_replace(F.col("production_year"), r"[^\d]", "").cast("integer"))

df = df.withColumn("engine_cc",
                   F.regexp_replace(F.col("engine_displacement"), r"[^\d]", "").cast("integer"))

df = df.withColumn("power_hp",
                   F.regexp_replace(F.col("power"), r"[^\d]", "").cast("integer"))

df = df.withColumn("fuel_clean",
                   F.lower(F.trim(F.col("fuel_type"))))

In [ ]:
df.select("vehicle_brand", "vehicle_model", "price_num", "mileage_km",
          "production_year_int", "engine_cc", "power_hp", "fuel_clean") \
  .show(10, truncate=False)

**Teoria:** `select()` to transformacja, która działa jak w SQL - pozwala wybrać podzbiór kolumn. Zmniejsza to ilość przetwarzanych danych w dalszych krokach.

In [ ]:
avg_brand = df.groupBy("vehicle_brand") \
              .agg(F.round(F.avg("price_num"), 2).alias("avg_price")) \
              .orderBy(F.col("avg_price").desc())

print("Średnia cena per marka")
avg_brand.show(20, truncate=False)

In [ ]:
fuel_count = df.groupBy("fuel_clean").count()
print("Liczba ogłoszeń wg rodzaju paliwa")
fuel_count.show()

In [ ]:
df.createOrReplaceTempView("cars")

In [ ]:
# SQL: zależność mocy i pojemności od ceny
spark.sql("""
    SELECT vehicle_brand,
           ROUND(AVG(power_hp), 1) AS avg_power,
           ROUND(AVG(engine_cc), 1) AS avg_cc,
           ROUND(AVG(price_num), 1) AS avg_price
    FROM cars
    GROUP BY vehicle_brand
    ORDER BY avg_power DESC
""").show()

In [ ]:
df.groupBy("production_year_int") \
  .count() \
  .orderBy(F.col("production_year_int").desc()) \
  .show()

In [ ]:
# Średnia cena i przebieg per marka i model
df.groupBy("vehicle_brand", "vehicle_model") \
  .agg(
      F.round(F.avg("price_num"), 2).alias("avg_price"),
      F.round(F.avg("mileage_km"), 2).alias("avg_mileage")
  ) \
  .orderBy(F.col("avg_price").desc()) \
  .show(20, truncate=False)

In [ ]:
# zależność ceny od przebiegu 
price_mileage = df.select("price_num", "mileage_km") \
                  .where((F.col("price_num").isNotNull()) & (F.col("mileage_km").isNotNull()))

In [ ]:
pdf_scatter = price_mileage.sample(fraction=0.1, seed=42).toPandas()

plt.figure(figsize=(8,5))
plt.scatter(pdf_scatter["mileage_km"], pdf_scatter["price_num"], s=6)
plt.title("Cena vs Przebieg")
plt.xlabel("Przebieg [km]")
plt.ylabel("Cena")
plt.tight_layout()
plt.savefig("scatter_price_mileage.png")

print("Wizualizacja scatter zapisana jako scatter_price_mileage.png")

**Teoria:** `toPandas()` to akcja, która zbiera (collect) wszystkie dane na partycjach roboczych i przesyła je na węzeł główny (Driver), konwertując do struktury Pandas DataFrame. Uwaga: Można tego używać tylko na małych zbiorach (po limitowaniu np. top 10), w przeciwnym razie braknie pamięci RAM na Driverze!

---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

In [ ]:
# Tutaj wpisz swój kod zliczający, czytający plik itp.
# Podpowiedź krok 1:
# df_crimes = spark.read.option("header", True).csv("chicago_crimes_sample.csv")
# df_crimes.show(5)

# Zadanie 1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, year, month, dayofmonth, hour,
    count, when, isnan
)

spark = SparkSession.builder \
    .appName("Chicago Crimes Analysis") \
    .getOrCreate()

spark

In [ ]:
df_raw = spark.read.csv(
    "/Workspace/Users/pichniarczykmarek@gmail.com/zad/chicago_crimes_sample.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)

df_raw.printSchema()
df_raw.show(5, truncate=False)

In [ ]:
raw_count = df_raw.count()

print(f"Liczba rekordów przed czyszczeniem: {raw_count}")

In [ ]:
duplicates_count = df_raw.count() - df_raw.dropDuplicates(["id"]).count()

print(f"Liczba duplikatów po kolumnie id: {duplicates_count}")

In [ ]:
df_no_duplicates = df_raw.dropDuplicates(["id"])

print(f"Liczba rekordów po usunięciu duplikatów: {df_no_duplicates.count()}")

In [ ]:
df_with_dates = df_no_duplicates \
    .withColumn("date_ts", to_timestamp(col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS")) \
    .withColumn("updated_on_ts", to_timestamp(col("updated_on"), "yyyy-MM-dd'T'HH:mm:ss.SSS"))

df_with_dates.select("date", "date_ts", "updated_on", "updated_on_ts").show(10, truncate=False)

In [ ]:
invalid_dates_count = df_with_dates.filter(col("date_ts").isNull()).count()

print(f"Liczba wierszy z błędną datą: {invalid_dates_count}")

In [ ]:
key_columns = [
    "id",
    "case_number",
    "date_ts",
    "primary_type",
    "location_description",
    "arrest",
    "domestic",
    "district",
    "year"
]

df_clean = df_with_dates.dropna(subset=key_columns)

print(f"Liczba rekordów po usunięciu braków w kluczowych kolumnach: {df_clean.count()}")

In [ ]:
df_clean = df_clean \
    .withColumn("year_from_date", year(col("date_ts"))) \
    .withColumn("month", month(col("date_ts"))) \
    .withColumn("day", dayofmonth(col("date_ts"))) \
    .withColumn("hour", hour(col("date_ts")))

In [ ]:
df_clean.filter(col("year") != col("year_from_date")) \
    .select("id", "date", "date_ts", "year", "year_from_date") \
    .show(20, truncate=False)

In [ ]:
df_clean = df_clean.drop("year").withColumnRenamed("year_from_date", "year")

In [ ]:
df_clean = df_clean.filter(
    (col("year") >= 2001) & 
    (col("year") <= 2026)
)

print(f"Liczba rekordów po odfiltrowaniu błędnych dat: {df_clean.count()}")

In [ ]:
df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in key_columns if c in df_clean.columns
]).show()

In [ ]:
df_clean.select(
    "id",
    "case_number",
    "date_ts",
    "primary_type",
    "description",
    "location_description",
    "arrest",
    "domestic",
    "district",
    "year",
    "hour"
).show(10, truncate=False)

In [ ]:
clean_count = df_clean.count()

print(f"Liczba rekordów przed czyszczeniem: {raw_count}")
print(f"Liczba rekordów po czyszczeniu: {clean_count}")
print(f"Usunięto rekordów: {raw_count - clean_count}")

# Zadanie 2

In [ ]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

In [ ]:
def classify_time_of_day(hour_value):
    if hour_value is None:
        return "brak danych"
    elif 0 <= hour_value <= 5:
        return "noc"
    elif 6 <= hour_value <= 11:
        return "rano"
    elif 12 <= hour_value <= 17:
        return "dzień"
    elif 18 <= hour_value <= 23:
        return "wieczór"
    else:
        return "niepoprawna godzina"

In [ ]:
classify_time_of_day_udf = udf(classify_time_of_day, StringType())

In [ ]:
df_clean = df_clean.withColumn(
    "time_of_day",
    classify_time_of_day_udf(col("hour"))
)

In [ ]:
df_clean.select(
    "date",
    "date_ts",
    "hour",
    "time_of_day",
    "primary_type",
    "location_description"
).show(20, truncate=False)

In [ ]:
df_clean.groupBy("time_of_day") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

# Zadanie 3

In [ ]:
from pyspark.sql.functions import broadcast, col, coalesce, lit

In [ ]:
# W klasycznym środowisku Spark można byłoby użyć cache():
# df_optimized = df_clean.cache()
# df_optimized.count()
#
# W Databricks serverless cache()/persist() nie jest obsługiwane,
# dlatego pracujemy bez fizycznego cache'owania.

df_optimized = df_clean

print(f"Liczba rekordów w df_optimized: {df_optimized.count()}")

### Uwaga dotycząca cache()

W klasycznym środowisku Spark można użyć `cache()` lub `persist()` po etapie czyszczenia danych, ponieważ oczyszczony DataFrame jest wykorzystywany wielokrotnie w dalszych analizach.

W tym przypadku notebook działa jednak na środowisku serverless, które nie obsługuje operacji `cache()` / `persist()`. Z tego powodu DataFrame `df_clean` został przypisany do `df_optimized` bez cache'owania.

Decyzja o miejscu użycia cache pozostaje jednak uzasadniona teoretycznie: najlepszym miejscem byłby moment po oczyszczeniu danych i dodaniu kolumn pomocniczych, a przed wykonywaniem wielu agregacji.

In [ ]:
crime_category_data = [
    ("THEFT", "przestępstwa przeciwko mieniu"),
    ("BATTERY", "przestępstwa przeciwko osobie"),
    ("CRIMINAL DAMAGE", "przestępstwa przeciwko mieniu"),
    ("ASSAULT", "przestępstwa przeciwko osobie"),
    ("MOTOR VEHICLE THEFT", "przestępstwa przeciwko mieniu"),
    ("ROBBERY", "przestępstwa przeciwko mieniu/osobie"),
    ("BURGLARY", "przestępstwa przeciwko mieniu"),
    ("NARCOTICS", "przestępstwa narkotykowe"),
    ("WEAPONS VIOLATION", "przestępstwa z użyciem broni"),
    ("DECEPTIVE PRACTICE", "oszustwa"),
    ("OTHER OFFENSE", "inne")
]

crime_category_df = spark.createDataFrame(
    crime_category_data,
    ["primary_type", "crime_category"]
)

crime_category_df.show(truncate=False)

In [ ]:
df_joined = df_optimized.join(
    broadcast(crime_category_df),
    on="primary_type",
    how="left"
)

df_joined = df_joined.withColumn(
    "crime_category",
    coalesce(col("crime_category"), lit("nieznana kategoria"))
)

df_joined.select(
    "id",
    "primary_type",
    "crime_category",
    "location_description",
    "time_of_day",
    "year"
).show(20, truncate=False)

In [ ]:
df_joined.explain(mode="formatted")

In [ ]:
df_partitioned = df_joined.repartition(4, "year")

In [ ]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.crimes_volume")

In [ ]:
spark.sql("SHOW VOLUMES IN workspace.default").show(truncate=False)

In [ ]:
output_path = "/Volumes/workspace/default/crimes_volume/chicago_crimes_parquet"

df_partitioned.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("year") \
    .save(output_path)

In [ ]:
df_parquet = spark.read.parquet(output_path)

df_parquet.printSchema()

df_parquet.groupBy("year") \
    .count() \
    .orderBy("year") \
    .show()

In [ ]:
print(f"Liczba rekordów przed zapisem: {df_joined.count()}")
print(f"Liczba rekordów po odczycie z Parquet: {df_parquet.count()}")

# Zadanie 4

In [ ]:
from pyspark.sql.functions import col, count, desc, row_number
from pyspark.sql.window import Window

In [ ]:
df_analysis = df_parquet

print(f"Liczba rekordów do analizy: {df_analysis.count()}")

In [ ]:
top_crime_types = df_analysis.groupBy("primary_type") \
    .agg(count("*").alias("crime_count")) \
    .orderBy(desc("crime_count"))

top_crime_types.show(15, truncate=False)

In [ ]:
top_locations = df_analysis.groupBy("location_description") \
    .agg(count("*").alias("crime_count")) \
    .orderBy(desc("crime_count"))

top_locations.show(15, truncate=False)

In [ ]:
crimes_by_time_of_day = df_analysis.groupBy("time_of_day") \
    .agg(count("*").alias("crime_count")) \
    .orderBy(desc("crime_count"))

crimes_by_time_of_day.show(truncate=False)

In [ ]:
crime_by_type_and_time = df_analysis.groupBy("time_of_day", "primary_type") \
    .agg(count("*").alias("crime_count")) \
    .orderBy("time_of_day", desc("crime_count"))

crime_by_type_and_time.show(40, truncate=False)

In [ ]:
crime_by_location_and_type = df_analysis.groupBy(
    "location_description",
    "primary_type"
).agg(
    count("*").alias("crime_count")
).orderBy(
    desc("crime_count")
)

crime_by_location_and_type.show(30, truncate=False)

In [ ]:
location_type_counts = df_analysis.groupBy(
    "location_description",
    "primary_type"
).agg(
    count("*").alias("crime_count")
)

In [ ]:
window_location = Window.partitionBy("location_description") \
    .orderBy(desc("crime_count"))

top_3_crimes_per_location = location_type_counts.withColumn(
    "rank",
    row_number().over(window_location)
).filter(
    col("rank") <= 3
).orderBy(
    "location_description",
    "rank"
)

top_3_crimes_per_location.show(60, truncate=False)

In [ ]:
crimes_by_location_and_time = df_analysis.groupBy(
    "location_description",
    "time_of_day"
).agg(
    count("*").alias("crime_count")
).orderBy(
    desc("crime_count")
)

crimes_by_location_and_time.show(40, truncate=False)

In [ ]:
heavy_aggregation = df_analysis.groupBy(
    "year",
    "month",
    "time_of_day",
    "location_description",
    "primary_type",
    "crime_category"
).agg(
    count("*").alias("crime_count")
).orderBy(
    desc("crime_count")
)

In [ ]:
heavy_aggregation.show(50, truncate=False)

In [ ]:
heavy_aggregation.explain(mode="formatted")